# Preprocessing: borders → Shades of Gray → hair removal

Three steps, each in a **window with sliders** (release a slider and the images update; `seed` changes the images).
**The steps are chained:** each window uses the output of the previous ones.

```
1. dark border removal  →  2. Shades of Gray (grey level)  →  3. hair removal (black-hat)
```

**How to use it**
1. Run all cells. Play with the 3 windows until you like the result. The values are saved automatically (`preprocessing/preproc_params.json`), and the windows reopen with them.
2. **Section 4** shows the values that will be used. You can also change any of them there by hand.
3. **Section 5** preprocesses **all** the images with those values and saves them to a new folder with the same structure, names and order.

If no window opens, change `%matplotlib tk` to `%matplotlib qt`.

In [1]:
%matplotlib tk
import sys, time
from pathlib import Path
from math import ceil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
import cv2

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import importlib, preprocessing
importlib.reload(preprocessing.preprocessing); importlib.reload(preprocessing)   # pick up edits to the .py
from preprocessing import (resize_image, fov_mask, remove_dark_borders, crop_black_borders_rect, correct_illumination,
                           hair_mask_stages, remove_hair, shades_of_gray, remove_specular,
                           gamma_correction, clahe_lab, denoise, add_noise, mse, psnr, ssim,
                           preprocess, config_from_params, HAIR_CHANNELS)

## 0. Configuration and data

In [8]:
CANDIDATES = [                       # first existing folder is used
    PROJECT_ROOT / "Binary" / "train",
    PROJECT_ROOT / "mini_dataset" / "train",
    PROJECT_ROOT / "mini_dataset" / "mini_dataset" / "train",
]
DATA_DIR = next((p for p in CANDIDATES if p.is_dir()), None)
assert DATA_DIR is not None, f"No image folder found, edit CANDIDATES. Tried: {CANDIDATES}"

SEED = 42            # initial seed of every window
PIN_IMAGES = []   # show THESE images (file names, without extension) instead of a random sample;
                             # [] = random sample with the seed slider
N_IMAGES = 5         # images per window (rows)
CLASS = "all"        # "all", "nevus", "others"
SHORT_SIDE = 256     # same size used by preprocess()
MAX_WINDOW_PX = 900  # window height limit (lower it on small screens)
PARAMS_FILE = preprocessing.preprocessing.PARAMS_FILE   # preprocessing/preproc_params.json (saved automatically)
LOAD_SAVED = True    # False = start every window from the default values

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
df = pd.DataFrame([{"path": f, "class": d.name.lower()}
                   for d in sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
                   for f in sorted(d.rglob("*")) if f.suffix.lower() in IMG_EXT])
print("Data:", DATA_DIR)
df["class"].value_counts()

Data: C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\mini_dataset\train


class
nevus     1017
others     983
Name: count, dtype: int64

### Helpers: sample, chain and window (`tuner`)
Parameter specs: `("int", lo, hi, init, step)`, `("float", lo, hi, init, step)`, `("choice", [options], init)`, `("bool", init)`.

In [3]:
# ---------------- sample ----------------
_SAMPLES = {}
def get_sample(seed):
    key = (seed, N_IMAGES, CLASS, SHORT_SIDE, tuple(PIN_IMAGES))
    if key not in _SAMPLES:
        if PIN_IMAGES:                                   # chosen images (seed is ignored)
            pick = df[df["path"].map(lambda p: p.stem in PIN_IMAGES)]
            missing = set(PIN_IMAGES) - {p.stem for p in pick["path"]}
            assert not missing, f"not found in {DATA_DIR}: {missing}"
            pick = pick.drop_duplicates(subset="path")
        else:
            sub = df if CLASS == "all" else df[df["class"] == CLASS]
            pick = sub.sample(n=min(N_IMAGES, len(sub)), random_state=seed)
        raw = [cv2.imread(str(p), cv2.IMREAD_COLOR) for p in pick["path"]]
        _SAMPLES[key] = dict(seed=seed, raw=raw, imgs=[resize_image(im, SHORT_SIDE) for im in raw],
                             labels=[f"{p.parent.name}/{p.name}" for p in pick["path"]])
    return _SAMPLES[key]

def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img

def overlay(img, mask, color=(0, 0, 255), alpha=0.65):
    out = img.copy()
    m = mask > 0
    out[m] = (alpha * np.array(color) + (1 - alpha) * out[m]).astype(np.uint8)
    return out

# ---------------- chain ----------------
CHAIN = ["borders", "color_constancy", "hair"]
PARAMS, USE, TUNERS = {}, {}, {}

def dn_kwargs(p):
    return dict(ksize=p["ksize"], bilateral_d=p["ksize"], sigma_color=p["sigma_color"],
                nlm_h=p["nlm_h"], nlm_h_color=p["nlm_h"], ad_iter=p["ad_iter"], ad_kappa=p["ad_kappa"])

def apply_step(step, img, p):
    if step == "borders":
        return resize_image(remove_dark_borders(img, **p), SHORT_SIDE)
    if step == "specular":
        return remove_specular(img, **p)
    if step == "illumination":
        return correct_illumination(img, **p)
    if step == "hair":
        return remove_hair(img, **p)
    if step == "color_constancy":
        return shades_of_gray(img, **p)
    if step == "denoise":
        return denoise(img, p["method"], **dn_kwargs(p))
    if step == "contrast":
        out = gamma_correction(img, p["gamma"])
        return clahe_lab(out, p["clip_limit"], p["tile_grid"]) if p["clahe"] else out
    raise ValueError(step)

_CACHE = {}
def run_chain(img, upto, key=None):
    '''Apply the active steps that come BEFORE `upto` (or all of them if upto=None).'''
    stop = CHAIN.index(upto) if upto else len(CHAIN)
    out = img
    for s in CHAIN[:stop]:
        if not USE.get(s):
            continue
        p = PARAMS[s]
        if key is None:
            out = apply_step(s, out, p)
            continue
        key = key + ((s, repr(sorted(p.items()))),)
        if key not in _CACHE:
            if len(_CACHE) > 500:
                _CACHE.clear()
            _CACHE[key] = apply_step(s, out, p)
        out = _CACHE[key]
    return out

def chain_inputs(S, step):
    '''Input of `step` for every image of the sample (cached).'''
    return [run_chain(img, step, key=(S["seed"], i, SHORT_SIDE)) for i, img in enumerate(S["imgs"])]

# Slider values are saved automatically to PARAMS_FILE every time a window updates,
# and a window (re)opened later starts from its last values (also after closing PyCharm).
import json
SAVED = json.loads(PARAMS_FILE.read_text()) if (LOAD_SAVED and PARAMS_FILE.exists()) else {}
if SAVED:
    print("Sliders start from the values saved in", PARAMS_FILE.name)

def _save_all():
    try:
        PARAMS_FILE.write_text(json.dumps(SAVED, indent=2))
    except OSError as e:
        print("could not save slider values:", e)

def _with_saved(name, specs):
    saved = SAVED.get(name, {})
    out = {}
    for k, sp in specs.items():
        if k in saved:
            v = saved[k]
            if sp[0] in ("int", "float"):
                sp = (sp[0], sp[1], sp[2], min(max(v, sp[1]), sp[2]), sp[4])
            elif sp[0] == "choice" and v in sp[1]:
                sp = (sp[0], sp[1], v)
            elif sp[0] == "bool":
                sp = (sp[0], bool(v))
        out[k] = sp
    return out

# ---------------- window ----------------
def _draw(ax, img, cmap=None, vmax=None):
    data = rgb(img)
    if ax.images and ax.images[0].get_array().shape == data.shape:
        im = ax.images[0]
        im.set_data(data)
    else:
        for im in list(ax.images):
            im.remove()
        im = ax.imshow(data)
        ax.set_xlim(-0.5, data.shape[1] - 0.5); ax.set_ylim(data.shape[0] - 0.5, -0.5)
    if data.ndim == 2:
        im.set_cmap(cmap or "gray"); im.set_clim(0, vmax or 255)

def tuner(name, params, compute, col_titles, step=None, use_default=True):
    '''Window with one row per image and one slider per parameter.
    compute(sample, **values) -> per image a list of cells: img | (img, title) | (img, title, dict(cmap, vmax)).
    step: name of the chain step this window controls (adds the `use_in_chain` slider).'''
    specs = {"seed": ("int", 0, 999, SEED, 1)}
    if step:
        specs["use_in_chain"] = ("bool", use_default)
    specs |= params
    specs = _with_saved(name, specs)
    n, ncols = (len(PIN_IMAGES) if PIN_IMAGES else N_IMAGES), len(col_titles)
    srows = ceil(len(specs) / 2)
    slider_px = srows * 30 + 55
    cell = int(max(110, min(240, (MAX_WINDOW_PX - slider_px - 50) / n)))
    W, H = max(ncols * cell + 130, 760), n * cell + slider_px + 50
    plt.close(name)
    fig = plt.figure(name, figsize=(W / 100, H / 100), dpi=100)
    gs = fig.add_gridspec(n, ncols, left=110 / W, right=1 - 10 / W, top=1 - 35 / H,
                          bottom=(slider_px + 5) / H, wspace=0.04, hspace=0.3)
    axes = [[fig.add_subplot(gs[r, c]) for c in range(ncols)] for r in range(n)]
    for row in axes:
        for ax in row:
            ax.set_xticks([]); ax.set_yticks([])

    getters, sliders, labelers = {}, {}, []
    for i, (k, sp) in enumerate(specs.items()):
        r, c = divmod(i, 2)
        y = (slider_px - 15 - (r + 1) * 30) / H
        ax = fig.add_axes([(c * W / 2 + 125) / W, y, (W / 2 - 190) / W, 20 / H])
        kind = sp[0]
        if kind in ("int", "float"):
            _, lo, hi, init, stp = sp
            s = Slider(ax, k, lo, hi, valinit=init, valstep=stp)
            getters[k] = (lambda s=s: int(round(s.val))) if kind == "int" else (lambda s=s: round(float(s.val), 4))
        else:
            opts = sp[1] if kind == "choice" else [False, True]
            init = sp[2] if kind == "choice" else sp[1]
            s = Slider(ax, k, 0, len(opts) - 1, valinit=opts.index(init), valstep=1)
            getters[k] = lambda s=s, o=opts: o[int(round(s.val))]
            lab = lambda s=s, o=opts: s.valtext.set_text(str(o[int(round(s.val))]))
            labelers.append(lab); lab()
        s.label.set_fontsize(8); s.valtext.set_fontsize(8)
        sliders[k] = s
    status = fig.text(0.01, 0.005, "", fontsize=7, color="0.3", wrap=True)
    state = {"dirty": False}

    def refresh(*_, cascade=True):
        vals = {k: g() for k, g in getters.items()}
        SAVED[name] = dict(vals)          # remember every slider (incl. seed) ...
        _save_all()                        # ... and write them to PARAMS_FILE
        sd = vals.pop("seed")
        if step:
            USE[step] = vals.pop("use_in_chain")
            PARAMS[step] = vals
        S = get_sample(sd)
        t0 = time.perf_counter()
        try:
            rows = compute(S, **vals)
        except Exception as e:
            status.set_text(f"ERROR: {type(e).__name__}: {e}"); status.set_color("r")
            fig.canvas.draw_idle(); return
        for r, row in enumerate(rows):
            for c, cellv in enumerate(row):
                cellv = cellv if isinstance(cellv, tuple) else (cellv, "")
                ax = axes[r][c]
                _draw(ax, cellv[0], **(cellv[2] if len(cellv) > 2 else {}))
                ax.set_title((col_titles[c] + "\n" if r == 0 else "") + cellv[1], fontsize=7)
            axes[r][0].set_ylabel(S["labels"][r], fontsize=7)
        status.set_color("0.3")
        status.set_text(f"{name}: {vals}   ({time.perf_counter() - t0:.1f} s, saved)")
        state["dirty"] = False
        fig.canvas.draw_idle()
        if cascade and step:              # update the windows that come after this step
            later = CHAIN[CHAIN.index(step) + 1:]
            for other, t in list(TUNERS.items()):
                if other != name and (t["step"] in later or t["step"] is None):
                    t["refresh"](cascade=False)

    def on_change(_):
        for lab in labelers:
            lab()
        state["dirty"] = True
    for s in sliders.values():
        s.on_changed(on_change)
    # recompute only when the mouse is released (not on every drag step)
    fig.canvas.mpl_connect("button_release_event", lambda e: refresh() if state["dirty"] else None)
    bax = fig.add_axes([1 - 95 / W, 8 / H, 85 / W, 24 / H])
    btn = Button(bax, "Update"); btn.label.set_fontsize(8)
    btn.on_clicked(lambda e: refresh())
    fig._keep = (sliders, btn)            # keep references alive
    TUNERS[name] = dict(fig=fig, refresh=refresh, sliders=sliders, step=step)
    refresh(cascade=False)
    plt.show(block=False)
    return TUNERS[name]

Sliders start from the values saved in preproc_params.json


## 1. Dark border removal
Many images have a **black frame or circular field of view**. Two methods:

* **`largest_rect` (default, Teresa's method).** Only dark areas **connected to the image edge** count as frame, so a very dark lesion in the middle is never removed. The image is cropped to the **largest rectangle without frame**. If that rectangle would be smaller than `min_area_frac` of the image, nothing is cropped.
* `crop_fill` / `fill` / `crop_inscribed`: the previous method. It detects the circle and fills the corners with the skin colour.

| Parameter | Method | What it does |
|---|---|---|
| `mode` | both | which method |
| `dark_thr` | largest_rect | gray level below which a pixel counts as frame (if it touches the edge) |
| `min_area_frac` | largest_rect | if the rectangle is smaller than this fraction of the image, no crop |
| `margin`, `edge` | circle modes | shrink the circle / where its border is placed |

Columns: input · frame detected (magenta) and crop (green) · output (resized again to `SHORT_SIDE`).


In [4]:
def borders_compute(S, mode, dark_thr, min_area_frac, margin, edge):
    p = dict(mode=mode, dark_thr=dark_thr, min_area_frac=min_area_frac, margin=margin, edge=edge)
    rows = []
    for img in S["imgs"]:
        vis = img.copy()
        if mode == "largest_rect":
            _, info = crop_black_borders_rect(img, dark_thr, min_area_frac, return_info=True)
            vis = overlay(vis, (~info["valid"]).astype(np.uint8) * 255, (255, 0, 255), 0.5)
            y0, y1, x0, x1 = info["box"]
            cv2.rectangle(vis, (x0, y0), (x1 - 1, y1 - 1), (0, 255, 0), 2)
            txt = "no crop (fallback)" if info["fallback"] else f"kept {100 * (y1 - y0) * (x1 - x0) / img.shape[0] / img.shape[1]:.0f}%"
        else:
            m = fov_mask(img, edge)
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(vis, cnts, -1, (0, 255, 0), 2)
            txt = "no frame" if m.min() > 0 else f"FOV {100 * (m > 0).mean():.0f}%"
        rows.append([img, (vis, txt), apply_step("borders", img, p)])
    return rows

tuner("borders", {
    "mode":          ("choice", ["largest_rect", "crop_fill", "fill", "crop_inscribed", "none"], "largest_rect"),
    "dark_thr":      ("int", 5, 100, 35, 1),
    "min_area_frac": ("float", 0.0, 0.5, 0.08, 0.01),
    "margin":        ("float", 0.0, 0.1, 0.02, 0.005),
    "edge":          ("float", 0.1, 0.9, 0.5, 0.05),
}, borders_compute, ["input", "frame / crop", "output"], step="borders");


## 2. Illumination normalisation: Shades of Gray (grey level)
**Input = output of window 1.** The light of each image is estimated with the Minkowski p-norm of each channel. With `normalize_brightness` **on** (default here), every image is scaled so that its light becomes the same **grey level** `target`: same tint and same overall brightness in all images. With it off, only the tint is corrected.
`power=1` is Gray World; a large power behaves like White Patch (p = 6 is the usual value in ISIC papers).

Columns: input · normalised · estimated light colour.

In [14]:
def cc_compute(S, power, normalize_brightness, target):
    rows = []
    for src in chain_inputs(S, "color_constancy"):
        out = shades_of_gray(src, power=power, normalize_brightness=normalize_brightness, target=target)
        illum = np.power(np.mean(np.power(src.astype(np.float32), power), axis=(0, 1)), 1.0 / power)
        sw = np.zeros((40, 40, 3), np.uint8); sw[:] = np.clip(illum, 0, 255).astype(np.uint8)
        b, g, r = illum
        rows.append([src, out, (sw, f"R{r:.0f} G{g:.0f} B{b:.0f}")])
    return rows

tuner("color_constancy", {
    "power":                ("int", 1, 20, 6, 1),
    "normalize_brightness": ("bool", True),
    "target":               ("float", 80.0, 240.0, 180.0, 5.0),
}, cc_compute, ["input (after 1)", "Shades of Gray", "estimated light"], step="color_constancy");

## 3. Hair removal: black-hat
**Input = output of windows 1 + 2 (borders removed, grey-level normalised).** Black-hat = closing − image is bright on thin dark structures narrower than the kernel (hairs). It is thresholded, round blobs are dropped, and the hair is filled from its surroundings (inpainting).

| Parameter | What it does | How to tune |
|---|---|---|
| `kernel_size` | must be **wider than the hairs** | thick hairs missed → increase; dots / lesion border detected → decrease |
| `thresh` | minimum black-hat value counted as hair | hairs missed → lower; skin texture detected → higher |
| `kernel_shape` | `cross` or `lines` (0/45/90/135°, DullRazor) | `lines` if dots/globules are taken as hair (slower) |
| `inpaint_radius` | neighbourhood used to fill the hair | smeared fill → lower |

`HAIR_ADVANCED = False` adds the secondary parameters. Only dark hairs are detected (white hairs would need a white top-hat).

In [9]:
HAIR_ADVANCED = False

_INP = {"telea": cv2.INPAINT_TELEA, "ns": cv2.INPAINT_NS}
hair_specs = {
    "kernel_size":    ("int", 3, 41, 17, 2),
    "thresh":         ("int", 1, 60, 10, 1),
    "kernel_shape":   ("choice", ["cross", "lines"], "cross"),
    "inpaint_radius": ("int", 1, 15, 5, 1),
}
if HAIR_ADVANCED:
    hair_specs |= {
        "channel":        ("choice", list(HAIR_CHANNELS), "gray"),
        "blur_ksize":     ("int", 1, 9, 3, 2),
        "filter_shape":   ("bool", True),
        "min_elongation": ("float", 1.0, 10.0, 3.0, 0.25),
        "min_area":       ("int", 0, 300, 20, 5),
        "max_fill":       ("float", 0.05, 1.0, 0.35, 0.05),
        "dilate_iter":    ("int", 0, 5, 1, 1),
        "inpaint_method": ("choice", ["telea", "ns"], "telea"),
    }

def hair_compute(S, inpaint_radius, inpaint_method="telea", **mask_kw):
    rows = []
    for img in chain_inputs(S, "hair"):
        st = hair_mask_stages(img, **mask_kw)
        clean = cv2.inpaint(img, st["mask"], inpaint_radius, _INP[inpaint_method])
        bh = st["blackhat"]
        rows.append([img,
                     (bh, f"max {bh.max()}", dict(cmap="magma", vmax=max(int(np.percentile(bh, 99.5)), mask_kw["thresh"] + 1))),
                     (overlay(img, st["mask"]), f"hair {100 * (st['mask'] > 0).mean():.1f}%"),
                     clean])
    return rows

tuner("hair", hair_specs, hair_compute, ["input (after 1+2)", "black-hat", "hair mask", "inpainted"], step="hair");

# 4. Chosen parameters
`CHOSEN` takes the values that are **currently in the windows** (or the last saved ones if you did not open the windows in this session).
To fix a value by hand, uncomment / add a line in the "manual changes" part, e.g. `CHOSEN["hair"]["thresh"] = 12`.

In [6]:
import pprint
STEPS = ["borders", "color_constancy", "hair"]
DEFAULTS = {
    "borders":         dict(mode="largest_rect", dark_thr=35, min_area_frac=0.08, margin=0.02, edge=0.5),
    "color_constancy": dict(power=6, normalize_brightness=True, target=195.0),
    "hair":            dict(kernel_size=5, thresh=15, kernel_shape="cross", inpaint_radius=5),
}
CHOSEN = {}
for s in STEPS:
    saved = {k: v for k, v in SAVED.get(s, {}).items() if k not in ("seed", "use_in_chain")}
    CHOSEN[s] = {**DEFAULTS[s], **saved, **PARAMS.get(s, {})}

# ---- manual changes (optional) ----
# CHOSEN["borders"]["margin"] = 0.03
# CHOSEN["color_constancy"]["target"] = 170
# CHOSEN["hair"]["thresh"] = 12
# -----------------------------------

FEATURE_PREPROC = dict(
    short_side=SHORT_SIDE,
    crop_borders=True,  crop_kwargs=CHOSEN["borders"],
    color_constancy=True,
    normalize_brightness=CHOSEN["color_constancy"]["normalize_brightness"],
    cc_kwargs={k: CHOSEN["color_constancy"][k] for k in ("power", "target")},
    hair=True, hair_kwargs=CHOSEN["hair"], hair_before_cc=False,   # order: borders -> SoG -> hair
)
pprint.pprint(CHOSEN, sort_dicts=False)

# quick check on 3 images: raw vs final
check = df.sample(3, random_state=SEED)["path"]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for j, p in enumerate(check):
    raw = cv2.imread(str(p))
    axes[0, j].imshow(rgb(resize_image(raw, SHORT_SIDE))); axes[0, j].set_title(p.name, fontsize=8)
    axes[1, j].imshow(rgb(preprocess(raw, **FEATURE_PREPROC))); axes[1, j].set_title("preprocessed", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show(block=False)

{'borders': {'mode': 'largest_rect',
             'dark_thr': 12,
             'min_area_frac': 0.04,
             'margin': 0.02,
             'edge': 0.4},
 'color_constancy': {'power': 6, 'normalize_brightness': True, 'target': 195.0},
 'hair': {'kernel_size': 5,
          'thresh': 15,
          'kernel_shape': 'cross',
          'inpaint_radius': 5}}


# 5. Preprocess all images and save them
Every image of every split (`train`, `val`, `test` if present) is processed with `FEATURE_PREPROC` and saved to `OUT_ROOT`, with **the same subfolders, file names and order**. The `*_labels.csv` files are copied unchanged, so they still match. A `params_used.json` file records the values used.
Images are processed in parallel with `N_WORKERS` threads (OpenCV releases the GIL, so threads really run at the same time).

`OUT_SHORT_SIDE`: size of the saved images (shorter side, px). Use `None` to keep the original resolution, which is slower, and with `None` the hair kernel works at a different scale than in the windows.

In [7]:
import json, os, shutil
from concurrent.futures import ThreadPoolExecutor

SRC_ROOT = DATA_DIR.parent                                   # e.g. mini_dataset/
OUT_ROOT = SRC_ROOT.parent / f"{SRC_ROOT.name}_preprocessed"  # e.g. mini_dataset_preprocessed/
OUT_SHORT_SIDE = SHORT_SIDE
JPEG_QUALITY = 95
OVERWRITE = True           # True = process again images that already exist in OUT_ROOT
N_WORKERS = os.cpu_count() or 4   # number of threads

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

cfg = {**FEATURE_PREPROC, "short_side": OUT_SHORT_SIDE}
splits = [d for d in sorted(SRC_ROOT.iterdir()) if d.is_dir() and d.name.lower() in ("train", "val", "test")]
print("source:", SRC_ROOT, "| splits:", [d.name for d in splits], "| output:", OUT_ROOT, "| threads:", N_WORKERS)

def process_one(f):
    '''Read -> preprocess -> write one image. Returns "done", "skipped" or "failed".'''
    dst = OUT_ROOT / f.relative_to(SRC_ROOT)                 # same subfolders and file name
    if dst.exists() and not OVERWRITE:
        return "skipped"
    img = cv2.imread(str(f), cv2.IMREAD_COLOR)
    if img is None:
        return "failed"
    out = preprocess(img, **cfg)
    dst.parent.mkdir(parents=True, exist_ok=True)
    params = [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY] if dst.suffix.lower() in (".jpg", ".jpeg") else []
    return "done" if cv2.imwrite(str(dst), out, params) else "failed"

# each thread processes whole images, so OpenCV's own internal threading is switched off
# meanwhile (otherwise threads compete for the same cores)
cv_threads = cv2.getNumThreads()
cv2.setNumThreads(1)
counts, failed, t0 = {"done": 0, "skipped": 0, "failed": 0}, [], time.time()
try:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for split in splits:
            files = sorted(f for f in split.rglob("*") if f.suffix.lower() in IMG_EXT)
            # ex.map returns results in the same order as `files`
            for f, status in zip(files, tqdm(ex.map(process_one, files), total=len(files), desc=split.name)):
                counts[status] += 1
                if status == "failed":
                    failed.append(str(f))
finally:
    cv2.setNumThreads(cv_threads)

for csv in SRC_ROOT.glob("*.csv"):                           # labels: same rows, same order
    shutil.copy2(csv, OUT_ROOT / csv.name)
(OUT_ROOT / "params_used.json").write_text(json.dumps({"CHOSEN": CHOSEN, "FEATURE_PREPROC": cfg}, indent=2))

print(f"{counts} in {time.time() - t0:.0f} s -> {OUT_ROOT}")
if failed:
    print("could not read / write:", failed[:10])

source: C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\mini_dataset | splits: ['train', 'val'] | output: C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\mini_dataset_preprocessed | threads: 8


train:   0%|          | 0/2000 [00:00<?, ?it/s]

val:   0%|          | 0/1000 [00:00<?, ?it/s]

{'done': 3000, 'skipped': 0, 'failed': 0} in 163 s -> C:\Users\Usuario\PycharmProjects\Melanoma-Challenge-ML-UdG\mini_dataset_preprocessed
